In [ ]:
# ====================== EDIT ONLY THIS BLOCK ======================
t = 1.0
lambda_nn = 0.5
lambda_nnn = 0.0
filling = 5/12
N = 144                 # 96 = faster; 144 = default; 192 = finer
eta = 0.06             # energy broadening for the nesting proxy
EF_override = None     # None -> fast numerical EF at target filling
# ================================================================

from pathlib import Path
import importlib.util, sys, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.path import Path as MplPath
from matplotlib.collections import LineCollection
from scipy.ndimage import map_coordinates

# Use the exact KagomeKaneMeleSOC Hamiltonian from your uploaded module.
model_path = Path('noninteracting.py')
if not model_path.is_file():
    raise FileNotFoundError('Put noninteracting.py next to this notebook.')
spec = importlib.util.spec_from_file_location('kagome_nonint_fast', model_path.resolve())
nonint = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = nonint
spec.loader.exec_module(nonint)

model = nonint.KagomeKaneMeleSOC(
    parameters={'t': float(t), 'l1': float(lambda_nn), 'l2': float(lambda_nnn)},
    spin=True,
)
# Its two spin sectors are conjugate and have identical dispersions / weights.
# We diagonalize just H_up (3x3) rather than the redundant spinful 6x6.
b1, b2 = np.asarray(model.b1), np.asarray(model.b2)

# Centered hexagon: same geometry used in your Figure 1 notebook.
bz = np.array([(2*b1-b2)/3, (b1+b2)/3, (-b1+2*b2)/3,
               -(2*b1-b2)/3, -(b1+b2)/3, -(-b1+2*b2)/3])
bz = bz[np.argsort(np.arctan2(bz[:,1],bz[:,0]))]

colors_ABC = np.array([[0.62,0.88,0.52], [0.95,0.56,0.13], [0.20,0.55,0.92]])
Bmat_inv = np.linalg.inv(np.column_stack((b1,b2)))

def at_xy(field, X, Y):
    """Bilinear periodic interpolation of a Bloch-spectrum/weight field."""
    uv = Bmat_inv @ np.vstack([np.ravel(X), np.ravel(Y)])
    out = map_coordinates(field, [N*uv[0], N*uv[1]], order=1,
                          mode='grid-wrap')
    return out.reshape(np.shape(X))

def bz_image(field, nplot=220):
    """Resample periodic torus data into the centered hexagonal Brillouin zone."""
    lo, hi = bz.min(axis=0), bz.max(axis=0)
    xs = np.linspace(lo[0], hi[0], nplot)
    ys = np.linspace(lo[1], hi[1], nplot)
    X, Y = np.meshgrid(xs, ys)
    inside = MplPath(bz).contains_points(np.column_stack([X.ravel(),Y.ravel()])).reshape(X.shape)
    return X, Y, inside, np.ma.masked_where(~inside, at_xy(field, X, Y))

def autocorr_fft(A):
    F = np.fft.fft2(A)
    return np.maximum(0.0, np.fft.ifft2(F.conj()*F).real / A.size)


In [ ]:
t0 = time.perf_counter()
# -------- ONE shared k-grid and eigensystem --------
uv = np.arange(N, dtype=float)/N
U, V = np.meshgrid(uv, uv, indexing='ij')
k = U[...,None]*b1 + V[...,None]*b2
H = np.stack([model.Hk_spin(float(p[0]),float(p[1])) for p in k.reshape(-1,2)]).reshape(N,N,3,3)
E, eigenvectors = np.linalg.eigh(H)              # (N,N,3), (N,N,3,3)
weights = np.abs(eigenvectors)**2                # (k1,k2,sublattice,band)

# Spin up/down have identical spectra: the per-state filling equals the
# 3-band filling, so a single 3-band block suffices to find EF.
flat = np.ptp(E, axis=(0,1))
span = float(np.ptp(E))
flat_at_EF = False
if EF_override is None:
    EF = float(np.quantile(E.ravel(), filling))
else:
    EF = float(EF_override)
for ib in range(3):
    if flat[ib] < max(1e-9, 1e-7*span) and abs(np.mean(E[...,ib])-EF) < max(1e-7,eta):
        flat_at_EF = True

# -------- Band dispersion Gamma -> K -> M -> Gamma --------
Gamma = np.zeros(2)
K = (b1+b2)/3
M = b2/2
path_points = [Gamma, K, M, Gamma]
ks, xp, ticks = [], [], [0.]
arc = 0.
for start, end in zip(path_points[:-1], path_points[1:]):
    step = float(np.linalg.norm(end-start))
    for f in np.linspace(0,1,85,endpoint=False):
        ks.append(start + f*(end-start)); xp.append(arc+f*step)
    arc += step; ticks.append(arc)
ks.append(Gamma); xp.append(arc)
Eb = np.array([np.linalg.eigvalsh(model.Hk_spin(*xy)) for xy in ks])

# -------- FS spectral weight and FFT-based nesting proxies --------
# gauss_eta ~ delta(E-E_F), normalized on the energy axis
G = np.exp(-0.5*((E-EF)/eta)**2)/(np.sqrt(2*np.pi)*eta)
A = np.sum(G, axis=-1)                                 # total FS spectral weight
Aorb = np.einsum('ijab,ijb->ija', weights, G)          # per-sublattice A/B/C
JDOS = autocorr_fft(A)
matched = sum(autocorr_fft(Aorb[...,a]) for a in range(3))
# 'matched' = same-sublattice weighted nesting, NOT the full Lindhard chi.

# -------- Figure --------
fig, axes = plt.subplots(2,2, figsize=(12,9), constrained_layout=True)
ax_band, ax_fs, ax_jdos, ax_match = axes.flat

for ib in range(3):
    ax_band.plot(xp,Eb[:,ib],color='#293449',lw=1.8)
ax_band.axhline(EF,ls='--',lw=1.3,color='#bf3e44',label=f'EF={EF:.4f}')
for x0 in ticks[1:-1]: ax_band.axvline(x0,c='0.8',lw=0.8)
ax_band.set_xticks(ticks, [r'$\Gamma$',r'$K$',r'$M$',r'$\Gamma$'])
ax_band.set_ylabel('Energy (t units when t=1)')
ax_band.set_title('(a) Band structure (spin up; spin down degenerate)')
ax_band.legend(frameon=False,loc='best',fontsize=9)
ax_band.margins(x=0)

# Obtain FS contours from the interpolated torus grid without any new diagonalizations.
X, Y, inside, _ = bz_image(E[...,0])
count = 0
for ib in range(3):
    vals = np.ma.masked_where(~inside, at_xy(E[...,ib],X,Y)-EF)
    if vals.min()>0 or vals.max()<0: continue
    # Extract contours; then color each segment by interpolated A/B/C weight.
    tmp = ax_fs.contour(X,Y,vals,levels=[0],linewidths=0)
    for line in tmp.allsegs[0]:
        if len(line)<2: continue
        mids=(line[:-1]+line[1:])/2
        ww=np.column_stack([at_xy(weights[...,a,ib], mids[:,0],mids[:,1]) for a in range(3)])
        ww=np.clip(ww,0,None); ww/=np.maximum(ww.sum(axis=1,keepdims=True),1e-15)
        lc=LineCollection(np.stack([line[:-1],line[1:]],axis=1),
                          colors=np.clip(ww@colors_ABC,0,1),linewidths=3.3)
        ax_fs.add_collection(lc); count+=1
    tmp.remove()
if flat_at_EF:
    # No regular 1D FS exists: a partially filled flat band lies at EF.
    ax_fs.text(0.5,0.05,'Flat band at EF: no isolated Fermi contour',
               transform=ax_fs.transAxes,ha='center',va='bottom',fontsize=9,
               bbox=dict(facecolor='white',alpha=.9,edgecolor='0.7'))
for a,name in enumerate('ABC'):
    ax_fs.scatter([],[],s=55,color=colors_ABC[a],label=name)
ax_fs.legend(ncol=3,loc='upper right',fontsize=9,frameon=False)
ax_fs.set_title('(b) FS colored by sublattice weight')

# Same color scale = directly comparable JDOS and orbital-matched intensities.
_,_,_,Jmap = bz_image(JDOS)
_,_,_,Smap = bz_image(matched)
upper = max(1e-12, float(np.percentile(Jmap.compressed(),99.5)))
for ax,data,title in [(ax_jdos,Jmap,'(c) Geometric nesting: FS JDOS'),
                      (ax_match,Smap,'(d) Same-sublattice-weighted nesting')]:
    pic=ax.pcolormesh(X,Y,data,cmap='magma',shading='auto',vmin=0,vmax=upper)
    fig.colorbar(pic,ax=ax,shrink=.79,label='Common intensity scale')
    ax.set_title(title)
    for qm in [b1/2,b2/2,(b1-b2)/2]:
        ax.plot(qm[0],qm[1],marker='o',mfc='none',mec='cyan',ms=5,mew=1.2)
    ax.plot(0,0,'+',color='cyan',ms=9,mew=1.4)

for ax in [ax_fs,ax_jdos,ax_match]:
    p=np.vstack([bz,bz[0]])
    ax.plot(p[:,0],p[:,1],color='0.5',lw=1)
    ax.set_aspect('equal'); ax.set_xlabel(r'$k_x$ / $q_x$')
    ax.set_ylabel(r'$k_y$ / $q_y$')
    ax.set_xlim(X.min(),X.max());ax.set_ylim(Y.min(),Y.max())

fig.suptitle(rf'Kagome: $t={t:g}$, $\lambda_{{NN}}={lambda_nn:g}$, '
             rf'$\lambda_{{NNN}}={lambda_nnn:g}$, filling={filling:.5f}, '
             rf'$N={N}$, $\eta={eta:g}$', fontsize=14)
plt.show()

print(f'EF = {EF:.7f}; band widths = {np.round(flat,6)}; '
      f'elapsed = {time.perf_counter()-t0:.2f} s')
for label,(u,v) in {'Gamma':(0,0),'M1':(.5,0),'M2':(0,.5),'M3':(.5,-.5)}.items():
    i,j = (int(round(u*N))%N, int(round(v*N))%N)
    print(f'{label:6s}: JDOS={JDOS[i,j]:.5g}, '
          f'orbital matched={matched[i,j]:.5g}, '
          f'ratio={matched[i,j]/max(JDOS[i,j],1e-15):.3f}')
if flat_at_EF:
    print('WARNING: A nearly flat band lies at EF (possibly partially filled).')
    print('FS is not a regular line; JDOS and orbital overlap are flat-band-dominated,')
    print('so interpret conventional nesting peaks with caution.')
print('Note: JDOS and same-sublattice overlap are fast nesting proxies,')
print('not the full Lindhard susceptibility, FRG response, or LCO susceptibility.')
